# Day 21 — What-if Inventory Simulator

Explore how lead time, review period, and service-level factor affect illustrative target stock.

This is a scenario simulator, not an operational recommendation. Rossmann sales are revenue-based, not unit demand.

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

input_path = DATA_DIR / "day16_safety_stock_reorder_point.csv"
if not input_path.exists():
    raise FileNotFoundError(f"Missing {input_path}. Run notebook 14 first.")
base = pd.read_csv(input_path)
required = ["Store","Average_Daily_Sales","Daily_Sales_Std"]
missing = [c for c in required if c not in base.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")
display(base.head())


,Store,Average_Daily_Sales,Daily_Sales_Std,Observed_Days,Lead_Time_Demand,Safety_Stock,Reorder_Point
0,1,4759.096031,1012.106393,781,33313.672215,4404.951088,37718.623303
1,2,4953.900510,1610.149102,784,34677.303571,7007.788990,41685.092562
2,3,6942.568678,2193.383804,779,48597.980745,9546.178585,58144.159329
3,4,9638.401786,1936.031881,784,67468.812500,8426.115869,75894.928369
4,5,4676.274711,1765.745628,779,32733.922978,7684.985669,40418.908647


## Configure scenario
Edit assumptions and rerun the next cell.

In [5]:
LEAD_TIME_DAYS = 7
REVIEW_PERIOD_DAYS = 7
Z_FACTOR = 1.645  # illustrative 95% one-sided service-level factor
ASSUMED_INVENTORY_POSITION = 0  # placeholder only

if LEAD_TIME_DAYS < 0 or REVIEW_PERIOD_DAYS < 0 or Z_FACTOR < 0:
    raise ValueError("Lead time, review period, and Z factor must be non-negative.")

scenario = base[["Store","Average_Daily_Sales","Daily_Sales_Std"]].copy()
scenario["Lead_Time_Demand"] = scenario["Average_Daily_Sales"] * LEAD_TIME_DAYS
scenario["Safety_Stock"] = Z_FACTOR * scenario["Daily_Sales_Std"] * np.sqrt(LEAD_TIME_DAYS)
scenario["Target_Stock"] = scenario["Average_Daily_Sales"] * (LEAD_TIME_DAYS + REVIEW_PERIOD_DAYS) + scenario["Safety_Stock"]
scenario["Assumed_Inventory_Position"] = ASSUMED_INVENTORY_POSITION
scenario["Recommended_Order_Quantity"] = (scenario["Target_Stock"] - scenario["Assumed_Inventory_Position"]).clip(lower=0)
display(scenario.head(10))


,Store,Average_Daily_Sales,Daily_Sales_Std,Lead_Time_Demand,Safety_Stock,Target_Stock,Assumed_Inventory_Position,Recommended_Order_Quantity
0,1,4759.096031,1012.106393,33313.672215,4404.951088,71032.295518,0,71032.295518
1,2,4953.900510,1610.149102,34677.303571,7007.788990,76362.396133,0,76362.396133
2,3,6942.568678,2193.383804,48597.980745,9546.178585,106742.140074,0,106742.140074
3,4,9638.401786,1936.031881,67468.812500,8426.115869,143363.740869,0,143363.740869
4,5,4676.274711,1765.745628,32733.922978,7684.985669,73152.831625,0,73152.831625
5,6,5509.946154,1577.380133,38569.623077,6865.169888,84004.416042,0,84004.416042
6,7,8817.050891,2448.618884,61719.356234,10657.028243,134095.740712,0,134095.740712
7,8,5539.358418,1894.094558,38775.508929,8243.593698,85794.611555,0,85794.611555
8,9,6562.337612,1628.653940,45936.363286,7088.326873,98961.053445,0,98961.053445
9,10,5568.420918,1051.132774,38978.946429,4574.804080,82532.696937,0,82532.696937


In [6]:
output_path = DATA_DIR / "day21_inventory_what_if.csv"
scenario.to_csv(output_path, index=False)
print(f"Saved scenario output to {output_path}")


Saved scenario output to ..\data\processed\day21_inventory_what_if.csv


## Conclusion
- Built a parameterized what-if calculation for lead time, review period, and service-level factor.
- Saved scenario outputs for later comparison.
- Change one assumption at a time to inspect the impact.
- Inventory position is a placeholder and sales are revenue-based; outputs are illustrative, not real purchasing recommendations.